# SCD Type 2 — Customer History

## Purpose

This notebook maintains historical versions of customer records in:

`migration.silver.customers_scd2`

It uses the current-state table:

`migration.silver.customers`

as the source.

### SCD Type 2 rule

For every `customer_id`:

- The old current row is closed with `effective_to = new updated_at`.
- The new row is inserted with `effective_from = new updated_at`.
- The new row has `effective_to = NULL`.
- Exactly **one** row must have `is_current = true`.

> **Important:** `migration.silver.customers` remains the current-state Silver table. This notebook creates and maintains a separate historical SCD2 table.


## Execution order

Run the notebook in this order:

1. Create the SCD2 table.
2. Perform the initial load **once**.
3. Validate the initial load.
4. Build the change set from current Silver.
5. Freeze/materialize the change set before modifying the target.
6. Close old current versions.
7. Validate that changed old versions are closed.
8. Insert new versions.
9. Run final SCD2 integrity checks.
10. Rebuild the change set and rerun it to prove idempotency.

For a real source change, the upstream PostgreSQL → Bronze → current Silver pipeline must run **before** Step 4.


# 1. Create the SCD2 table

This is safe to run repeatedly because `IF NOT EXISTS` does not recreate an existing table.


In [0]:
%sql
CREATE TABLE IF NOT EXISTS migration.silver.customers_scd2
(
    customer_id BIGINT,
    customer_name STRING,
    email STRING,
    city STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    -- SCD Type 2 validity columns
    effective_from TIMESTAMP,
    effective_to TIMESTAMP,
    is_current BOOLEAN,

    -- Pipeline lineage
    _source_batch_id STRING,
    _source_run_id STRING,
    _source_ingestion_timestamp TIMESTAMP,
    _processed_timestamp TIMESTAMP
)
USING DELTA;


# 2. Inspect the SCD2 table

On a brand-new table, the expected result is zero rows.

If the table already contains data, **do not run the initial-load INSERT again**. Continue to the validation/change-detection sections.


In [0]:
%sql
SELECT *
FROM migration.silver.customers_scd2
ORDER BY customer_id, effective_from;


# 3. Initial SCD2 load — run once

The current Silver table represents the latest known state of each customer.

For the initial SCD2 load, each customer becomes the first/current version:

- `effective_from = updated_at`
- `effective_to = NULL`
- `is_current = TRUE`


In [0]:
%sql
INSERT INTO migration.silver.customers_scd2
SELECT
    customer_id,
    customer_name,
    email,
    city,
    created_at,
    updated_at,

    updated_at AS effective_from,
    CAST(NULL AS TIMESTAMP) AS effective_to,
    TRUE AS is_current,

    _source_batch_id,
    _source_run_id,
    _source_ingestion_timestamp,
    current_timestamp() AS _processed_timestamp

FROM migration.silver.customers s
WHERE NOT EXISTS
(
    SELECT 1
    FROM migration.silver.customers_scd2 t
    WHERE t.customer_id = s.customer_id
);


# 4. Validate the initial SCD2 load

Expected after your current 11-customer baseline:

- `total_records = 11`
- `current_records = 11`
- `historical_records = 0`

The exact counts depend on the current contents of `migration.silver.customers`.


In [0]:
%sql
SELECT
    COUNT(*) AS total_records,
    COUNT_IF(is_current = true) AS current_records,
    COUNT_IF(is_current = false) AS historical_records
FROM migration.silver.customers_scd2;


## 4.1 Validate the one-current-row rule

Expected: **0 rows**.

If this query returns anything, stop. The SCD2 table is invalid and should not be processed further until it is repaired.


In [0]:
%sql
SELECT
    customer_id,
    COUNT(*) AS version_count,
    COUNT_IF(is_current = true) AS current_count
FROM migration.silver.customers_scd2
GROUP BY customer_id
HAVING COUNT_IF(is_current = true) != 1;


# 5. Build the change set

This compares the latest/current Silver record with the current SCD2 version.

A row is considered changed when:

`source.updated_at > target.updated_at`

If there is no source change, this should return zero rows.

This logic intentionally handles **updates detected by the watermark pipeline**. It does not detect hard deletes from PostgreSQL.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW scd2_changes AS

SELECT
    s.*
FROM migration.silver.customers s
INNER JOIN migration.silver.customers_scd2 t
    ON s.customer_id = t.customer_id
   AND t.is_current = true
WHERE s.updated_at > t.updated_at;


In [0]:
%sql
SELECT
    customer_id,
    customer_name,
    email,
    city,
    updated_at,
    _source_batch_id,
    _source_run_id
FROM scd2_changes
ORDER BY customer_id;


# 6. Freeze the change set before modifying the target

This is important.

`scd2_changes` is a view derived from the target table. After we close current target rows, the result of that view could change if it is evaluated again.

Cache/materialize the change set **before** the MERGE and use that frozen result for both the close and insert steps.

#### Cache is not working in Serverless so we use TEMP VIEW


In [0]:
%sql

CREATE OR REPLACE TEMP TABLE scd2_changes_frozen AS

SELECT
    customer_id,
    customer_name,
    email,
    city,
    created_at,
    updated_at,
    _source_batch_id,
    _source_run_id,
    _source_ingestion_timestamp

FROM scd2_changes;

In [0]:
%sql

SELECT
    customer_id,
    customer_name,
    city,
    updated_at
FROM scd2_changes_frozen
ORDER BY customer_id;

## Checkpoint

If there are no source changes:

`scd2_changes` should contain **0 rows**.

If you intentionally changed a customer, such as customer 7 from Nashik → Mumbai, that customer should appear here.

**Do not continue if the change set is not what you expect.**


# 7. Close the old current versions

For every detected change:

- Find the current SCD2 row.
- Set `effective_to` to the new source `updated_at`.
- Set `is_current = false`.

This does **not** insert the new version yet.


In [0]:
%sql
MERGE INTO migration.silver.customers_scd2 AS target

USING scd2_changes_frozen AS source

ON target.customer_id = source.customer_id
AND target.is_current = true

WHEN MATCHED
AND source.updated_at > target.updated_at

THEN UPDATE SET
    target.effective_to = source.updated_at,
    target.is_current = false;


# 8. Validate the old versions were closed

For each changed customer, there should now be no current row left from the old version.

The query below should return the changed customers with `is_current = false`.


In [0]:
%sql
SELECT
    customer_id,
    customer_name,
    city,
    updated_at,
    effective_from,
    effective_to,
    is_current
FROM migration.silver.customers_scd2
WHERE customer_id IN (
    SELECT customer_id FROM scd2_changes
)
ORDER BY customer_id, effective_from;


# 9. Insert the new current versions

Now create the new SCD2 version for each detected change.

The new version has:

- `effective_from = updated_at`
- `effective_to = NULL`
- `is_current = TRUE`

The `NOT EXISTS` condition is an additional guard against inserting the same version twice.


In [0]:
%sql
INSERT INTO migration.silver.customers_scd2
SELECT
    s.customer_id,
    s.customer_name,
    s.email,
    s.city,
    s.created_at,
    s.updated_at,

    s.updated_at AS effective_from,
    CAST(NULL AS TIMESTAMP) AS effective_to,
    TRUE AS is_current,

    s._source_batch_id,
    s._source_run_id,
    s._source_ingestion_timestamp,
    current_timestamp() AS _processed_timestamp

FROM scd2_changes_frozen s

WHERE NOT EXISTS
(
    SELECT 1
    FROM migration.silver.customers_scd2 t
    WHERE t.customer_id = s.customer_id
      AND t.updated_at = s.updated_at
      AND t.is_current = true
);


# 10. Verify the changed customer(s)

This should show two versions for a customer that changed once:

```text
old version → is_current = false
new version → is_current = true
```

For example, after customer 7 changes Nashik → Mumbai, the expected pattern is:

```text
7 | Vikas More | Nashik | ... | 2026-09-03... | false
7 | Vikas More | Mumbai | ... | NULL           | true
```


In [0]:
%sql
SELECT
    customer_id,
    customer_name,
    city,
    updated_at,
    effective_from,
    effective_to,
    is_current
FROM migration.silver.customers_scd2
WHERE customer_id IN (
    SELECT customer_id FROM scd2_changes
)
ORDER BY customer_id, effective_from;


# 11. Final SCD2 integrity validation

## Rule: exactly one current version per customer

Expected: **0 rows**.


In [0]:
%sql
SELECT
    customer_id,
    COUNT(*) AS version_count,
    COUNT_IF(is_current = true) AS current_count
FROM migration.silver.customers_scd2
GROUP BY customer_id
HAVING COUNT_IF(is_current = true) != 1;


In [0]:
%sql
SELECT
    COUNT(*) AS total_records,
    COUNT_IF(is_current = true) AS current_records,
    COUNT_IF(is_current = false) AS historical_records
FROM migration.silver.customers_scd2;


# 12. Validate effective-date continuity

For every historical row, `effective_to` should be populated.

The query should return **0 rows**.


In [0]:
%sql
SELECT
    customer_id,
    city,
    effective_from,
    effective_to,
    is_current
FROM migration.silver.customers_scd2
WHERE is_current = false
  AND effective_to IS NULL;


# 13. Validate current rows have no end date

The query should return **0 rows**.

A current version must have:

`effective_to IS NULL`


In [0]:
%sql
SELECT
    customer_id,
    city,
    effective_from,
    effective_to,
    is_current
FROM migration.silver.customers_scd2
WHERE is_current = true
  AND effective_to IS NOT NULL;


# 14. Prove SCD2 idempotency

After the new version has been inserted, the old cached change set is no longer appropriate for a new run.

First remove the cache, then rebuild the change set from the current target state.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW scd2_changes AS

SELECT
    s.*
FROM migration.silver.customers s
INNER JOIN migration.silver.customers_scd2 t
    ON s.customer_id = t.customer_id
   AND t.is_current = true
WHERE s.updated_at > t.updated_at;


In [0]:
%sql
SELECT
    customer_id,
    customer_name,
    city,
    updated_at
FROM scd2_changes
ORDER BY customer_id;


### Expected result

If PostgreSQL/current Silver has not changed since the SCD2 update:

**0 rows**

That proves the SCD2 process does not create another version when there is no new source change.


# 15. Final SCD2 history view

Use this query when demonstrating the result in an interview.

It shows the complete history chronologically.


In [0]:
%sql
SELECT
    customer_id,
    customer_name,
    city,
    effective_from,
    effective_to,
    is_current,
    _source_batch_id,
    _source_run_id
FROM migration.silver.customers_scd2
ORDER BY customer_id, effective_from;


# 16. How the complete pipeline works

```text
PostgreSQL
    |
    | updated_at watermark
    v
Bronze (append-only history)
    |
    v
Current Silver
migration.silver.customers
    |
    | compare current state
    v
SCD2 change detection
    |
    +---- changed? ---- no ----> do nothing
    |
   yes
    |
    v
Close old version
is_current = false
effective_to = new updated_at
    |
    v
Insert new version
is_current = true
effective_from = new updated_at
effective_to = NULL
    |
    v
SCD2 History
migration.silver.customers_scd2
```

## Key distinction

`migration.silver.customers` answers:

> **What is the customer's current state?**

`migration.silver.customers_scd2` answers:

> **What was the customer's state over time?**

The upstream process is still **watermark-based incremental ingestion**, not source-log CDC. SCD Type 2 is the historical modeling layer built on top of that incremental pipeline.
